# 00 — Data Ingestion

**Owner:** Aparna (Data Lead) · **Support:** Rolando (Data Pipeline & QA)

Merges and normalizes the three approved sources (Fitzpatrick17k, SKINCON, Google SCIN) into one dataset: `data/processed/manifest.csv` + `data/external/label_map.csv`. See `docs/data-strategy.md`.

This notebook is a thin, reproducible wrapper over the library code in `src/dermaface/data/` — the logic lives there (not in the notebook) so it is tested and reusable. If no raw data is present it generates a small synthetic slice so the notebook always runs.

In [ ]:
import sys, os
from pathlib import Path
import pandas as pd
ROOT = Path.cwd()
while not (ROOT / 'src' / 'dermaface').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src')); sys.path.insert(0, str(ROOT / 'scripts'))
from dermaface.config import CLASS_NAMES, SPLIT_NAMES, load_config
from dermaface.data import download, manifest, splits
print('classes:', CLASS_NAMES, '| splits:', SPLIT_NAMES)

## 1. Locate / validate the raw sources

`download_source` validates that each source's metadata CSV is present in `data/raw/<source>/` (it does not re-download). Missing images are fine at this stage — the manifest is metadata.

In [ ]:
cfg = load_config()
raw_dir = cfg.data_dir / 'raw'
have_raw = (raw_dir / 'fitzpatrick17k' / 'fitzpatrick17k.csv').exists()
if not have_raw:
    print('No raw CSVs found — generating a synthetic slice for this demo run.')
    os.environ.setdefault('DERMAFACE_DATA_DIR', str(ROOT / 'data_smoke'))
    from make_smoke_slice import make_slice
    cfg = load_config(); raw_dir = cfg.data_dir / 'raw'
    make_slice(raw_dir, per_class=12)
for src in download.SOURCES:
    try:
        download.download_source(src, raw_dir, download_images=False)
        print('validated:', src)
    except FileNotFoundError as e:
        print('missing:', src, '->', str(e)[:100])

## 2. Build the label map (crosswalk)

Every distinct raw condition string per source, and the class it maps to (or `dropped`). Auditable, and the input to Notebook 02's taxonomy work.

In [ ]:
lm_path, n = manifest.build_label_map(raw_dir)
lm = pd.read_csv(lm_path)
print(f'{n} distinct raw labels | kept {(lm.action=="kept").sum()} | dropped {(lm.action=="dropped").sum()}')
display(lm[lm.action=='kept'].sort_values('n_rows', ascending=False).head(12))

## 3. Build the manifest

Harmonizes the sources into `manifest.csv` with schema `path,label,severity,skin_type,source,split`.

In [ ]:
manifest_path, summary = manifest.build_manifest(raw_dir, require_image_exists=False)
print(summary.render())
df = pd.read_csv(manifest_path)
display(df.head())

## 4. Stratified splits (train/eval/test/demo)

Stratified by (label, skin_type); test frozen; demo reserved for the app.

In [ ]:
counts = splits.make_splits(manifest_path)
print('split counts:', counts)
df = pd.read_csv(manifest_path)
display(pd.crosstab(df['label'], df['split']).reindex(index=CLASS_NAMES, columns=SPLIT_NAMES, fill_value=0))

## 5. Handoff

- EDA and image-quality plots: `01_eda.ipynb`.
- QA report: `make qa` → `data/processed/qa/data_quality_report.csv`.
- Taxonomy + severity method: `02_label_harmonization.ipynb` (Aparna + Iva).

### Member contributions (Sprint 1, data)
- **Aparna** — data strategy, source selection & licensing, label taxonomy.
- **Rolando** — ingestion/download pipeline, manifest + label_map, splits, QA, tests.